In [1]:
import os
from dotenv import load_dotenv
from src.agent import run_agent
from src.memory import SessionMemory
from src.hooks import get_logs
from src.tools import web_search, read_file
import pandas as pd

load_dotenv()
print("Ready.")

Ready.


In [2]:
memory = SessionMemory()
print("Memory initialized.")
print(f"Current facts: {memory.get_facts()}")

Memory initialized.
Current facts: No facts stored yet.


In [3]:
print("Testing web search tool directly...\n")
result = web_search("Chelsea FC latest news 2026")
print(result)

Testing web search tool directly...

Title: Summer transfers 2026: All the Chelsea ins, outs and new contracts so far | News | Official Site | Chelsea Football Club
Summary: With the 2026 summer transfer window now open, we keep you updated with all the deals done so far this summer... The Italian international became our first signing of the summer when he joined on a permanent deal from Atalanta at the start of July. Palestra signed a contract through to 2033. The winger&#x27;s move to Chelsea was first confirmed in March 2025, before officially completing his move from Sporting Lisbon this summer.
URL: https://www.chelseafc.com/en/news/article/summer-transfers-2026-all-the-chelsea-ins-outs-and-new-contracts-so-far

Title: Chelsea Fixtures, Results & Standings 2026/27 | Premier League
Summary: For the latest news on Chelsea, including scores, fixtures, team news, results, form guide &amp; league position, visit the official website of the Premier League.
URL: https://www.premierleagu

In [4]:
print("Testing file read tool directly...\n")
result = read_file("files/chelsea.txt")
print(result)

Testing file read tool directly...

Chelsea FC Facts

Chelsea Football Club was founded in 1905.
They are based in Fulham, London at Stamford Bridge stadium.
Chelsea have won the Premier League title 6 times.
They won the UEFA Champions League in 2012 and 2021.
Notable managers include Jose Mourinho, Carlo Ancelotti, and Thomas Tuchel.
Their nickname is The Blues.
Their rival clubs are Arsenal, Tottenham, and Manchester United.



In [5]:
print("=" * 50)
print("MULTI-HOP AGENT DEMO")
print("=" * 50)

questions = [
    "Read the file files/chelsea.txt and tell me when Chelsea FC was founded",
    "Search the web for Chelsea FC current manager in 2026",
    "Based on what you know from memory, how many Champions League titles has Chelsea won and who was a notable manager?",
]

for i, question in enumerate(questions, 1):
    print(f"\nQuestion {i}: {question}")
    print("-" * 40)
    answer = run_agent(question, memory)
    print(f"Answer: {answer}")
    print()

MULTI-HOP AGENT DEMO

Question 1: Read the file files/chelsea.txt and tell me when Chelsea FC was founded
----------------------------------------
Agent thinking...
[Hook] 2026-07-15 15:30:50 | Calling tool: read_file with args: {'filepath': 'files/chelsea.txt'}
[Hook] 2026-07-15 15:30:50 | Tool read_file completed. Preview: Chelsea FC Facts

Chelsea Football Club was founded in 1905.
They are based in Fulham, London at Sta...
Answer: According to the file, **Chelsea Football Club was founded in 1905**.


Question 2: Search the web for Chelsea FC current manager in 2026
----------------------------------------
Agent thinking...
[Hook] 2026-07-15 15:30:53 | Calling tool: web_search with args: {'query': 'Chelsea FC current manager 2026'}
[Hook] 2026-07-15 15:30:54 | Tool web_search completed. Preview: Title: Chelsea Football Club is delighted to announce the appointment of Xabi Alonso as Manager of t...
[Memory] Stored: Searched for: Chelsea FC current manager 2026 - found results
Answer

In [6]:
print("Facts stored in memory after the conversation:\n")
print(memory.get_facts())

Facts stored in memory after the conversation:

- Searched for: Chelsea FC current manager 2026 - found results


In [7]:
logs = get_logs()
print(f"Total tool calls logged: {len(logs)}\n")
df = pd.DataFrame(logs)
df

Total tool calls logged: 4



,timestamp,event,tool,args,result_preview
0,2026-07-15 15:30:50,tool_call,read_file,{'filepath': 'files/chelsea.txt'},NaN
1,2026-07-15 15:30:50,tool_result,read_file,NaN,Chelsea FC Facts\n\nChelsea Football Club was ...
2,2026-07-15 15:30:53,tool_call,web_search,{'query': 'Chelsea FC current manager 2026'},NaN
3,2026-07-15 15:30:54,tool_result,web_search,NaN,Title: Chelsea Football Club is delighted to a...


## Week 4 Findings

### Agent Architecture
The agent follows a planner-executor-memory loop:
1. User sends a message
2. Agent (LLM) decides which tool to use
3. Tool executes and returns result
4. Hook logs the call with timestamp
5. Result goes back to agent
6. Agent decides if more tools are needed or answers directly

### Tools
Two tools were implemented:
- web_search uses the Brave Search API to find current information
- read_file reads .txt and .pdf files from disk

### Memory
SessionMemory stores facts extracted during the conversation.
When the agent searches for something it stores what it searched for.
On the next question the agent receives these facts in its system
prompt and can reference them without searching again.

### Hooks
Pre and post tool hooks run before and after every tool call.
They log the tool name, arguments, timestamp, and result preview
to a log file. This is useful for debugging and auditing.

### Multi-Hop Reasoning
The agent answered the third question using memory from the first
two questions without calling any tools again. This demonstrates
that the agent can chain reasoning across multiple turns.